In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 49.4 MB/s eta 0:00:00


In [2]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets

print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

✅ CUDA available: True
✅ GPU: Tesla T4


In [3]:
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR = "./codellama-coding-chat"

MAX_SEQ_LENGTH = 1024
CODE_SAMPLES = 15000
CHAT_SAMPLES = 6000

print(f"📁 Output: {OUTPUT_DIR}")
print(f"📊 Samples: {CODE_SAMPLES} code + {CHAT_SAMPLES} chat = {CODE_SAMPLES + CHAT_SAMPLES} total")

📁 Output: ./codellama-coding-chat
📊 Samples: 15000 code + 6000 chat = 21000 total


In [4]:
print("\n🚀 Loading datasets to CPU...")

print("Loading code dataset...")
code_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
code_dataset = code_dataset.shuffle(seed=42).select(range(min(CODE_SAMPLES, len(code_dataset))))
print(f"✅ Code: {len(code_dataset)} examples")

print("Loading chat dataset...")
chat_dataset = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train")
chat_dataset = chat_dataset.shuffle(seed=42).select(range(min(CHAT_SAMPLES, len(chat_dataset))))
print(f"✅ Chat: {len(chat_dataset)} examples")


🚀 Loading datasets to CPU...
Loading code dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

✅ Code: 15000 examples
Loading chat dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44245 [00:00<?, ? examples/s]

✅ Chat: 6000 examples


In [5]:
def format_code(example):
    instruction = example.get('prompt', example.get('instruction', ''))
    input_text = example.get('input', '')
    output = example.get('completion', example.get('output', ''))

    if input_text:
        prompt = f"{instruction}\n\nInput: {input_text}"
    else:
        prompt = instruction

    return f"[INST] {prompt.strip()} [/INST] {output.strip()}"

def format_chat(example):
    prompt = example.get('prompt', example.get('question', example.get('instruction', '')))
    response = example.get('chosen', example.get('response', example.get('answer', example.get('output', ''))))

    if isinstance(response, list):
        response = response[0] if response else ""

    return f"[INST] {prompt.strip()} [/INST] {str(response).strip()}"

print("✅ Format functions ready")

✅ Format functions ready


In [6]:
print("\n⚙️  Formatting datasets (CPU processing)...")

code_formatted = code_dataset.map(
    lambda x: {"text": format_code(x)},
    remove_columns=code_dataset.column_names,
    desc="Formatting code"
)

chat_formatted = chat_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=chat_dataset.column_names,
    desc="Formatting chat"
)

full_dataset = concatenate_datasets([code_formatted, chat_formatted])
full_dataset = full_dataset.shuffle(seed=42)

print(f"✅ Total: {len(full_dataset)} examples")
print(f"📖 Sample: {full_dataset[0]['text'][:150]}...")


⚙️  Formatting datasets (CPU processing)...


Formatting code:   0%|          | 0/15000 [00:00<?, ? examples/s]

Formatting chat:   0%|          | 0/6000 [00:00<?, ? examples/s]

✅ Total: 21000 examples
📖 Sample: [INST] Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Interpret the resul...


In [ ]:
print("\n🚀 Loading model to GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded!")
print(f"📊 Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"💾 GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


🚀 Loading model to GPU...


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]